**SA234 &#x25aa; Data Wrangling and Visualization &#x25aa; Spring 2026**

# Lesson 9. Interactive Visualization

## In this lesson...

- Visualization can be a powerful way to make sense of data


- A single image, however, can only do so much


- Through *interaction*, we can transform static images into tools for exploration

- For example, we can let the user:
    - highlight points of interest
    - zoom in to reveal finer-grained patterns
    - link across multiple charts to see multi-dimensional relationships

- In this lesson, we will learn about Altair's tools for enabling this kind of interaction

<hr style="border-top: 2px solid gray; margin-top: 1px; margin-bottom: 1px"></hr>

## Easy mode: tooltips

- Let's start by importing Pandas and Altair, as usual:

In [ ]:
import pandas as pd
import altair as alt

- For this section, we'll use the Gapminder data that we've used in previous lessons, located in the same folder as this notebook, in `data/gapminder.csv`


- Let's start with a simple, but very useful mode of interaction: *tooltips*


- A **tooltip** is an informational text box that appears when hovering over a particular part of a chart 


- Consider the following chart, similar to ones we've created before:

In [ ]:
gap_df = pd.read_csv('data/gapminder.csv')

In [ ]:
alt.Chart(gap_df).transform_filter(
    'datum.year == 2000'
).mark_circle(opacity=0.5).encode(
    alt.X('fertility:Q').title('Number of children per woman'),
    alt.Y('life_expect:Q').title('Average life expectancy (years)'),
    alt.Size('pop:Q').scale(range=[0, 1000]),
    alt.Color('cluster:N')
).properties(
    width=600,
    height=400
)

- There's a lot of information conveyed in this chart


- However, we can't tell which country corresponds to each circle


- We could use text labels, but that might be too cluttered


- Instead, we can use tooltips to display this information


- The `Tooltip` encoding channel determines the text to show when a user moves the mouse over a mark


- Let's map the `country` variable to the `Tooltip` encoding:

- We can include multiple pieces of information in a tooltip by passing a *list* of `Tooltip` encodings to `.encode()`


- We can also include titles for each tooltip applying the method `.title()` to `alt.Tooltip()`


- For example, let's include population in our tooltips, in addition to the country name:

- This is great, but there's a small problem


- You might noticed that as you mouse around, tooltips don't appear for some of the points


- For example, the circle corresponding to India is drawn on top of a country with a smaller population, preventing the mouse from hovering over that country


- To fix this problem, we can use the `Order` encoding channel

- The `Order` encoding channel determines the order of data points, affecting:
    - the order in which they are drawn and, 
    - for `line` and `area` marks, the order in which they are connected to one another

- Let's order the values in descending order of population, ensuring that smaller circles are drawn later than larger circles:

- Now we can identify the smaller country being obscured by India!

❓ **Exercise 1.**
Modify the chart above so the tooltips also include the average life expectancy and number of children per woman.

- This is a great first step... how else can we make our Altair charts interactive?

<hr style="border-top: 2px solid gray; margin-top: 1px; margin-bottom: 1px"></hr>

## Another easy mode: panning and zooming

- The points in the scatter plot we created above are rather small and dense in places


- We can let the user inspect these places more closely by enabling *panning* and *zooming*

- **Panning** is the ability to move the chart view left and right 
    - Panning is done by clicking-and-dragging in the chart area with the left mouse button

- **Zooming** is the ability to change the magnification level of the chart view
    - Zooming is done using the mouse scroll wheel

- We can enable both panning and zooming with the `.interactive()` method of a Chart object, like this:

In [ ]:
alt.Chart(gap_df).transform_filter(
    'datum.year == 2000'
).mark_circle(opacity=0.5).encode(
    alt.X('fertility:Q').title('Number of children per woman'),
    alt.Y('life_expect:Q').title('Average life expectancy (years)'),
    alt.Size('pop:Q').scale(range=[0, 1000]),
    alt.Color('cluster:N'),
    [alt.Tooltip('country:N').title('Country'),
     alt.Tooltip('pop:Q').title('Population')]
).properties(
    width=600,
    height=400
)

❓ **Exercise 2.**
[Here is the Altair documentation for the `alt.Chart` object](https://altair-viz.github.io/user_guide/generated/toplevel/altair.Chart.html). Look for the `.interactive()` method in this documentation.
How can you restrict the panning and zooming to only the x-axis? How about the y-axis?

*Write your answer here. Double-click to edit.*

<hr style="border-top: 2px solid gray; margin-top: 1px; margin-bottom: 1px"></hr>

## A brief interlude on Altair syntax

### Specifying encodings using keyword arguments

- Up to this point in the course, we've primarily used the encoding *classes*, like `alt.X()` and `alt.Y()`, to specify encodings:

In [ ]:
alt.Chart(gap_df).mark_circle().encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Color('cluster:N')
)

- Recall that we can equivalently use the encoding *keyword arguments*, like `x=...` and `y=...`:

In [ ]:
alt.Chart(gap_df).mark_circle().encode(
    x='fertility:Q',
    y='life_expect:Q',
    color='cluster:N'
)

- We'll need to use this keyword argument syntax when specifying *conditions* for interaction

### Specifying mark properties through encodings

- We can set various mark properties through the appropriate `.mark_...()` method:

In [ ]:
alt.Chart(gap_df).mark_circle(color='red').encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q')
)

- We can also set these properties through the corresponding encoding keyword arguments and `alt.value()`:

In [ ]:
alt.Chart(gap_df).mark_circle().encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    color=alt.value('red')
)

<hr style="border-top: 2px solid gray; margin-top: 1px; margin-bottom: 1px"></hr>

## Hard mode: parameters and conditions

- With Altair, we can create more advanced ways to interact with our visualizations

- Altair provides a grammar for interaction:
    - **Parameters** are the basic building blocks for interaction
        - A parameter can either be a simple variable, or a mapping between user input (e.g., mouse clicks) and a subset of observations
    - **Conditions and filters** can update chart elements based on changes in parameter values 
    - **Bindings and widgets** allow parameter values to be changed via drop-down menus, radio buttons, sliders, etc. 

## A small example

- Let's start with a small example


- We'll use a dataset containing information on 406 car models manufactured between 1970 and 1982 ([source](http://lib.stat.cmu.edu/datasets/)), located in the same folder as this notebook in `data/cars.csv`

In [ ]:
cars_df = pd.read_csv('data/cars.csv')
cars_df.head()

- Let's work on adding interactivity to the following scatter plot of horsepower vs. miles per gallon:

In [ ]:
alt.Chart(cars_df).mark_circle().encode(
    alt.X('Miles_per_Gallon:Q').title('Miles per gallon'),
    alt.Y('Horsepower:Q').title('Horsepower'),
    alt.Color('Origin:N').title('Country of origin')
)

### Variable parameters

- Variable parameters allow for a value to be defined once and then reused throughout the rest of the chart

- We can create a variable parameter using the `alt.param()` function

- For example, we can set the opacity of the circle marks using a variable parameter like this:

- We can then bind the variable parameter `op_var` to a widget so that the user can change its value dynamically 

- For example, we can introduce a slider widget with the function `alt.binding_range()` and bind it to `op_var` like this 

### Selection parameters

- Selection parameters define subsets of observations through interactive manipulation of the chart by the user 

- For example, let's create an **interval selection**, which allows the user to select chart elements by clicking and dragging

- To do this, we use `alt.selection_interval()`, like this:

- Next, we can register the `interval` selection with our chart with the `.add_params()` method:

- Now, we have a chart that allows the user to click and drag to create a selection region

### Conditions

- We can make our chart respond to a selection using a **condition**

- In general, we specify a condition with the `alt.condition()` function, like this:

    ```python
    encoding_name=alt.condition(
        selection,
        encoding_specification_if_in_selection,
        encoding_specification_if_not_in_selection
    )
    ```

- Let's make our chart
    - color the *selected* points based on their values of `Origin`, and
    - color the other points `'lightgray'`

- Now, when we select a region of points, only those points are colored based on the value of `Origin` 😎

<hr style="border-top: 2px solid gray; margin-top: 1px; margin-bottom: 1px"></hr>

## Selection types

- Again, recall that a selection parameter maps user input (e.g., mouse clicks) into a subset of observations


- Now that we have a basic idea of how interaction works in Altair, let's dig a bit deeper into the available selection types

### Interval selection

- An **interval selection** allows the user to select a continuous range of chart elements by clicking and dragging
    

- We create such selections using `alt.selection_interval()`


- We used an interval selection in our small example earlier:

In [ ]:
interval = alt.selection_interval()

alt.Chart(cars_df).mark_circle().encode(
    alt.X('Miles_per_Gallon:Q').title('Miles per gallon'),
    alt.Y('Horsepower:Q').title('Horsepower'),
    color=alt.condition(
        interval, 
        alt.Color('Origin:N').title('Country of origin'), 
        alt.value('lightgray')
    )
).add_params(
    interval
)

- By default, the resulting subset of observations for an interval selection is the set of observations whose marks fall within the click-and-dragged box

- The `alt.selection_interval()` function takes several optional keyword arguments
    - [Altair documentation for `alt.selection_interval()`](https://altair-viz.github.io/user_guide/generated/api/altair.selection_interval.html)

- Let's try a more complex example

- The `encodings=...` keyword argument of `alt.selection_interval()` specifies a list of encodings
    - The selection's resulting subset of observations must have encoding values that match the click-and-dragged box 

- For example, if we specify `encodings=['x']`, then the selection's resulting subset of observations must have x-axis values that match the click-and-dragged box


- Let's see this in action:

### Point selections

- A **point selection** allows the user to select chart elements one at a time with the mouse


- We create such selections using `alt.selection_point()`


- By default, chart elements are selected on click


- For example, let's modify the chart above to highlight a single point when the user clicks on it:

- Multiple chart elements can be selected by clicking on them while holding the <kbd>Shift</kbd> key

- By default, the resulting subset of observations for a point selection is the observations corresponding to the clicked elements 

- The `alt.selection_point()` function also takes several optional keyword arguments
    - [Altair documentation for `alt.selection_point()`](https://altair-viz.github.io/user_guide/generated/api/altair.selection_point.html)

❓ **Exercise 3.**
Look at the documentation for [`alt.selection_point()`](https://altair-viz.github.io/user_guide/generated/api/altair.selection_point.html).
Use keyword arguments for `alt.selection_point()` to modify the chart above so that

- the user select points when mousing over them (*Hint.* The "Vega event stream" needed is `'mouseover'`)
- the user has a bit of leeway and always selects the points nearest to the mouse pointer
- the chart starts with none of the observations in the selection

<hr style="border-top: 2px solid gray; margin-top: 1px; margin-bottom: 1px"></hr>

## Bindings and widgets

- In all of the examples above, selections were used to *directly* choose marks on a chart and the corresponding subset of observations
    - This is the default behavior of selections

- We can also **bind** selections to chart elements (e.g. legends) and **widgets** (e.g. dropdown menus)

### Widget binding 

- Altair supports several types of input elements:

| Input Element            | Description                                           |
| :-                       | :-                                                    |
| `alt.binding_select()`   | Dropdown menu for selecting a single item from a list |
| `alt.binding_radio()`    | Radio buttons that force only a single selection      |
| `alt.binding_range()`    | Slider to allow selection along a scale               |
| `alt.binding_checkbox()` | Checkboxes allowing multiple selections of items      |

#### Data-driven lookups 

- Data-driven lookups use the active value(s) of the widget together with a selection parameter to look up points with matching values in the chart


- Let's add a dropdown menu to our scatter plot of horsepower vs. miles per gallon, so that we can highlight the observations corresponding to cars from a specific region:

- Let's break this code down...

- First, we use `alt.binding_select()` to create a dropdown menu
    - We specify `options` corresponding to the values in the `Origin` variable in the dataset
    - We give a `name` that will be used to label the dropdown menu in the chart

- Next, we use `alt.selection_point()` to create a point selection
    - We specify a list of `fields` (i.e., variables) whose values we want to match
    - In this example, we match observations based on their value of `Origin`
    - We `bind` the selection to the dropdown menu we created earlier
    - We also specify a default `value` of `'Europe'` to match with the first entry of the dropdown menu 
   

- Finally, the rest of the chart is constructed the same way as before 


- Whew! 🤯

- Instead of coloring the marks based on the selection, we can perform a *filter transform* based on the selection instead, like this:

#### Combining multiple parameters from multiple widgets 

- We can also create a selection based on multiple parameters from multiple widgets, so that we can highlight observations based on the values of multiple variables

- For example, we can modify our chart to highlight the observations corresponding to cars made in particular years with specific numbers of cylinders

- First, we'll set up sliders for both the number of cylinders and the year:

In [ ]:
cylinder_slider = alt.binding_range(
    min=4, max=8, step=1,
    name='Cylinders: '
)

year_slider = alt.binding_range(
    min=1970, max=1982, step=1,
    name='Year: '
)

cylinder_select = alt.selection_point(
    fields=['Cylinders'],
    bind=cylinder_slider,
    value=4
)

year_select = alt.selection_point(
    fields=['Year'],
    bind=year_slider,
    value=1970
)

- Then, we can specify the parameters and condition in our chart as follows:

- In the code above, note the use of `&` in `alt.condition()` to tell Altair that we want to color the points that match _both_ `cylinder_select` and `year_select` parameters

- You can use the following logical composition operands:

| Operand | Meaning |
| :- | :- |
| `&` | and |
| `\|` | or |
| `~` | not |


#### Data-driven comparisons

- So far, we have seen how to use selections to look up points with _exactly_ matching values in the chart

- Sometimes we want to make a more complex comparison than an exact match (e.g., less than, greater than)

- For example, we can modify our chart to highlight the observations corresponding to cars with a weight below a certain level

- First, we'll set up a slider for the weight:

In [ ]:
weight_slider = alt.binding_range(
    min=1000, max=6000, step=10,
    name='Weight below: '
)

weight_var = alt.param(
    bind=weight_slider,
    value=1500
)

- Then, we can specify the parameter and condition in our chart as follows:

- In the chart above, note the use of `alt.param()` to define the variable parameter `weight_var`

- This variable parameter `weight_var` is then used in `alt.condition()` to specify the comparison between the variable `Weight_in_lbs` and the value of `weight_var`

- Also note the use of the special syntax `alt.datum.variable_name` to reference the variable being compared in `alt.condition()`

### Legend binding

- Depending on your visualization, it might be appropriate to *use the legend itself* as a selection mechanism


- For example, we can create a chart that highlights observations based on their country of origin when we click on the appropriate part of the legend, like this:

- We can actually select multiple parts of the legend simultaneously with <kbd>Shift</kbd>-click


- The keyword argument `bind='legend'` binds the multiple selection to the legend


- The keyword argument `fields=['Origin']` matches observations based on values of `Origin`

<hr style="border-top: 2px solid gray; margin-top: 1px; margin-bottom: 1px"></hr>

## Dynamic queries through visualization

- Now, we can get really fancy...

- We can go beyond the standard input elements provided by Altair and use a *visualization itself* as an interface for dynamic queries

- Let's start with the scatter plot below, and:
    - add a histogram of the cars by weight, and 
    - use the *histogram* as a means to select points on the scatter plot

In [ ]:
scatter = alt.Chart(cars_df).mark_circle().encode(
    alt.X('Miles_per_Gallon:Q').title('Miles per gallon'),
    alt.Y('Horsepower:Q').title('Horsepower'),
    alt.Color('Origin:N').title('Country of origin')
).properties(
    width=600,
    height=400
)

scatter

- The example above provides dynamic queries using a _linked selection_ between charts:
    - We created an interval selection (`brush`), and set `encodings=['x']` to limit the selection to the x-axis only, resulting in a one-dimensional selection interval
    - We register `brush` with our histogram of cars by weight with `.add_params(brush)`.
    - We use `brush` in a conditional encoding to adjust the color of the scatter plot

<hr style="border-top: 2px solid gray; margin-top: 1px; margin-bottom: 1px"></hr>

## Problems

### Problem 1

For this problem, you will use the movies dataset that we've used in previous lessons. You can find it in the same folder as this notebook, in `data/movies.csv`.

Create a scatter plot of the Rotten Tomatoes ratings vs. the IMDB ratings. Make your chart interactive by having a tooltip appear when the user mouses over each point, displaying the title of the movie and the year of the movie's release date.

### Problem 2

Modify your chart from Problem 1 so that it allows the user to highlight the points in the chart corresponding to a particular genre.

In particular:


- Use color to differentiate the different genres, and include an accompanying legend.


- Configure the legend so that when the user clicks on a movie genre, the points corresponding to movies of that genre are highlighted in your chart. In particular, set the opacity of the points corresponding to that genre to 0.75, and the opacity of all other points to 0.05.


- Enable pan-and-zoom.

### Problem 3

Modify your chart from Problem 1 so that it allows the user to highlight the points in the chart corresponding to a particular genre.

In particular:

- Create a dropdown menu, with options corresponding to the different genres in the dataset. In the code cell below, there is a list called `genres` with these genres.


- When the user selects a genre from the dropdown menu, set the opacity of the points corresponding to that genre to 0.75, and the opacity of all other points to 0.05.


- Set the initial selection value of the dropdown menu to `'Drama'`.


- Enable pan-and-zoom.

In [ ]:
genres = ['Drama', 'Comedy', 'Musical', 'Thriller/Suspense',
          'Adventure', 'Action', 'Romantic Comedy', 'Horror', 'Western',
          'Documentary', 'Black Comedy', 'Concert/Performance']

### Problem 4

Now, modify your chart from Problem 2 so that it allows the user to highlight points in the chart corresponding to a particular genre *and* MPAA rating.

In particular:

- In addition to the dropdown menu you created in Problem 2, create a set of radio buttons that the user can use to select an MPAA rating. In the code cell below, there is a list called `mpaa` with these ratings.


- When the user selects a genre from the dropdown menu and an MPAA rating from the radio buttons, set the opacity of the points corresponding to that combination to 0.75, and the opacity of all other points to 0.05.


- Set the initial selection value of the dropdown menu to `'Drama'`, and the radio buttons to `'R'`.


- Enable pan-and-zoom.


*Bonus.* Are there any G- or PG-rated horror movies? 

In [ ]:
mpaa = ['G', 'PG', 'PG-13', 'R', 'NC-17', 'Not Rated']

<hr style="border-top: 2px solid gray; margin-top: 1px; margin-bottom: 1px"></hr>

## Notes and sources

- These lesson notes are based on the [Visualization Curriculum](https://uwdata.github.io/visualization-curriculum/) by the University of Washington


- [Altair documentation on interactive charts](https://altair-viz.github.io/user_guide/interactions.html)
